In [0]:
base_path = "/Volumes/teaching/datasets/assignment"
print(base_path)

/Volumes/teaching/datasets/assignment


In [0]:
# dbutils.fs.head(f"{base_path}/emails.csv")

In [0]:
# import csv
# import sys
# csv.field_size_limit(sys.maxsize)
# fp = open(f"{base_path}/emails.csv", "r")
# reader = csv.reader(fp)
# for line in reader:
#     print(line)
#     print('---------\n\n')
# fp.close()

In [0]:
df = spark.read.csv(
    f"{base_path}/emails.csv",
    header=True,
    inferSchema=True,
    quote='"',
    escape='"',
    multiLine=True,
)

In [0]:
random_emails = df.sample(False, 0.0001).limit(10).collect()

In [0]:
for i, row in enumerate(random_emails, start=1):
    print('--'*50)
    print(f'Row #{i}')
    print('--'*50)
    print(row["message"])
    print()

----------------------------------------------------------------------------------------------------
Row #1
----------------------------------------------------------------------------------------------------
Message-ID: <19084532.1075854584231.JavaMail.evans@thyme>
Date: Wed, 22 Nov 2000 03:04:00 -0800 (PST)
From: newsletter@investorama.com
To: ebass@enron.com
Subject: INSIDE INVESTORAMA 11.22.00
Mime-Version: 1.0
Content-Type: text/plain; charset=us-ascii
Content-Transfer-Encoding: 7bit
X-From: Inside Investorama <newsletter@investorama.com>
X-To: Dominator128 <ebass@enron.com>
X-cc: 
X-bcc: 
X-Folder: \Eric_Bass_Dec2000\Notes Folders\All documents
X-Origin: Bass-E
X-FileName: ebass.nsf

I N S I D E   I N V E S T O R A M A                    November 22, 2000
Your Portal to Financial Freedom
http://www.investorama.com/

Dear Dominator128,

Welcome to "Inside Investorama," your weekly email newsletter bringing you 
the best of Investorama this week. Check out the latest features, new 

In [0]:
messages_df = df.select("message").cache()
total_number_of_emails = messages_df.count()  # Force materialization to break Volume lineage

In [0]:
for i, row in enumerate(messages_df.take(10)):
    print(f'Row #{i}')
    print('--'*50)
    message = row["message"]
    print((message))
    print

Row #0
----------------------------------------------------------------------------------------------------
Message-ID: <18782981.1075855378110.JavaMail.evans@thyme>
Date: Mon, 14 May 2001 16:39:00 -0700 (PDT)
From: phillip.allen@enron.com
To: tim.belden@enron.com
Subject: 
Mime-Version: 1.0
Content-Type: text/plain; charset=us-ascii
Content-Transfer-Encoding: 7bit
X-From: Phillip K Allen
X-To: Tim Belden <Tim Belden/Enron@EnronXGate>
X-cc: 
X-bcc: 
X-Folder: \Phillip_Allen_Jan2002_1\Allen, Phillip K.\'Sent Mail
X-Origin: Allen-P
X-FileName: pallen (Non-Privileged).pst

Here is our forecast

 
Row #1
----------------------------------------------------------------------------------------------------
Message-ID: <15464986.1075855378456.JavaMail.evans@thyme>
Date: Fri, 4 May 2001 13:51:00 -0700 (PDT)
From: phillip.allen@enron.com
To: john.lavorato@enron.com
Subject: Re:
Mime-Version: 1.0
Content-Type: text/plain; charset=us-ascii
Content-Transfer-Encoding: 7bit
X-From: Phillip K Allen
X-

In [0]:
import re

def extract_from_email(email_text):
    print(email_text)
    match = re.search(r'^From:\s*(.+)$', email_text, re.MULTILINE)
    return match.group(1).strip() if match else None

def extract_to_emails(email_text):
    # Match "To:" and everything until the next header field using positive lookahead
    match = re.search(r'^To:\s*(.*?)(?=^\S+:)', email_text, re.MULTILINE | re.DOTALL)
    if not match:
        return None # return None so we can filter with isNotNull()
    
    to_field = match.group(1)
    
    # Extract all email addresses from the field
    emails = re.findall(r'[\w\.-]+@[\w\.-]+', to_field)
    return emails

def extract_subject_line(email_text):
    match = re.search(r'Subject:\s*(.*?)\s+\w+:', email_text)
    if not match:
        return None
    subject = match.group(1).strip()
    return subject if subject else None

# Question A1

What is the mean number of emails sent by each sender in the dataset, i.e., the number of emails sent divided by the
number of unique senders in the dataset (based on the ‘From’ field)

The mean number of emails sent by each sender.
1. Get all the senders
2. Filter out duplicate senders
3. Get all messages sent
4. Calculate the mean

In [0]:
from_list = messages_df.rdd.map(lambda x: extract_from_email(x["message"]))

In [0]:
print(from_list.count(), from_list)

517401 PythonRDD[373] at RDD at PythonRDD.scala:62


In [0]:
unique_senders = from_list.distinct()
count_unique_senders = unique_senders.count()

In [0]:
mean_number_of_emails_by_each_sender = total_number_of_emails/count_unique_senders
print(mean_number_of_emails_by_each_sender)

25.45262691853601


# Question A2

What is the total number of unique recipients in the dataset (based on the ‘To’ field, you can filter out emails which don’t contain this field, or which don’t appear to contain a valid email address in this field)

In [0]:
to_list = (messages_df.rdd
    .map(lambda x: extract_to_emails(x["message"]))
    .filter(lambda x: x is not None)  # filter out None
)
display(to_list)

PythonRDD[389] at RDD at PythonRDD.scala:62

In [0]:
unique_recipients = to_list.flatMap(lambda x: x).distinct()
count_unique_recipients = unique_recipients.count()
print(count_unique_recipients)

70785


In [0]:
# from collections import Counter

# def get_top_n_words(email_text, n):
#     words = re.findall(r'\w+', email_text)
#     word_counts = Counter(words)
#     top_n_words = word_counts.most_common(n)
#     return top_n_words
# top_n_words = messages_df.rdd.map(lambda x: get_top_n_words(x["message"], 10))
# print(top_n_words.collect())

# Question A3

What is the number of emails with a subject line (i.e., the 'Subject' field is not missing or empty)?

In [0]:
subject_list = (
    messages_df.rdd
    .map(lambda x: extract_subject_line(x["message"]))
    .filter(lambda x: x is not None and x != "")
)
count_subjects = subject_list.count()
print(count_subjects)

148413


In [0]:
# sample_1 = """
# Message-ID: <18782981.1075855378110.JavaMail.evans@thyme>
# Date: Mon, 14 May 2001 16:39:00 -0700 (PDT)
# From: phillip.allen@enron.com
# To: tim.belden@enron.com
# Subject: 
# Mime-Version: 1.0
# Content-Type: text/plain; charset=us-ascii
# Content-Transfer-Encoding: 7bit
# X-From: Phillip K Allen
# X-To: Tim Belden <Tim Belden/Enron@EnronXGate>
# X-cc: 
# X-bcc: 
# X-Folder: \Phillip_Allen_Jan2002_1\Allen, Phillip K.\'Sent Mail
# X-Origin: Allen-P
# X-FileName: pallen (Non-Privileged).pst
# """
# print(extract_subject_line(sample_1))

In [0]:
# sample_2 = """
# Nick Hiemstra
# 09/18/2000 04:55 PM
# To: Eric Bass/HOU/ECT@ECT
# cc:  
# Subject: Re: GAME WEDNEDSDAY @ 7:00  

# I was sweating in the 3rd qtr. when we were down 17-14.  What do you think 
# they'll do about the QB situation?

# I do have to say, our QB question has definitely been answered.  I just hope 
# our defense can hold up.  The Tech and Colorado games should be good guides.

# """

# print(extract_subject_line(sample_2))

# Question B1

Who are the top 10 senders by number of emails sent (based on the ‘From’ field), along with the number of emails they each sent

In [0]:
from collections import Counter

# Most common subject lines
Counter(from_list.collect()).most_common(10)

[('kay.mann@enron.com', 16735),
 ('vince.kaminski@enron.com', 14368),
 ('jeff.dasovich@enron.com', 11411),
 ('pete.davis@enron.com', 9149),
 ('chris.germany@enron.com', 8801),
 ('sara.shackleton@enron.com', 8777),
 ('enron.announcements@enron.com', 8587),
 ('tana.jones@enron.com', 8490),
 ('steven.kean@enron.com', 6759),
 ('kate.symes@enron.com', 5438)]

In [0]:
from pyspark.sql.functions import exists, udf, explode, col
from pyspark.sql.types import ArrayType, BooleanType, StringType

In [0]:
extract_from_udf = udf(extract_from_email, StringType())

top_senders = (
    messages_df
    .withColumn("from_email", extract_from_udf("message"))
    .groupBy("from_email")
    .count()
    .orderBy("count", ascending=False)
    .limit(10)
)
display(top_senders)

from_email,count
kay.mann@enron.com,16735
vince.kaminski@enron.com,14368
jeff.dasovich@enron.com,11411
pete.davis@enron.com,9149
chris.germany@enron.com,8801
sara.shackleton@enron.com,8777
enron.announcements@enron.com,8587
tana.jones@enron.com,8490
steven.kean@enron.com,6759
kate.symes@enron.com,5438


# Question B2

Who are the top 10 recipients by number of emails received (based on the ‘To’ field, filtering out emails not containing this info), along with the number of emails they received

In [0]:
extract_to_udf = udf(extract_to_emails, ArrayType(StringType()))

top_receivers = (
    messages_df
    .withColumn("to_email_list", extract_to_udf("message"))
    .filter(col("to_email_list").isNotNull()) # filter out emails with no To field
    .withColumn("to_email", explode("to_email_list"))  # one row per recipient
    .groupBy("to_email")
    .count()
    .orderBy("count", ascending=False)
    .limit(10)
)

display(top_receivers)

to_email,count
richard.shapiro@enron.com,15149
jeff.dasovich@enron.com,14207
tana.jones@enron.com,12828
steven.kean@enron.com,12754
sara.shackleton@enron.com,11433
james.steffes@enron.com,10347
mark.taylor@enron.com,9787
pete.davis@enron.com,9281
susan.mara@enron.com,9064
paul.kaufman@enron.com,8522


# Question B3

What is the number of emails sent internally within Enron (based on both 'From' and 'To' fields containing @enron.com)

The approach here is to filter for `From` fields that contain `@enron.com` and also filter for `To` email list that has at least one element with `@enron.com`

In [0]:
has_enron_udf = udf(lambda emails: any("@enron.com" in e for e in emails) if emails else False, BooleanType())

internal_emails = (
    messages_df
    .withColumn("to_email_list", extract_to_udf("message"))
    .filter(col("to_email_list").isNotNull())
    .filter(has_enron_udf("to_email_list"))          # at least one @enron.com in To
    .withColumn("from_email", extract_from_udf("message"))
    .filter(col("from_email").contains("@enron.com")) # From contains @enron.com
)

In [0]:
display(internal_emails)

message,to_email_list,from_email
"Message-ID: <18782981.1075855378110.JavaMail.evans@thyme> Date: Mon, 14 May 2001 16:39:00 -0700 (PDT) From: phillip.allen@enron.com To: tim.belden@enron.com Subject: Mime-Version: 1.0 Content-Type: text/plain; charset=us-ascii Content-Transfer-Encoding: 7bit X-From: Phillip K Allen X-To: Tim Belden X-cc: X-bcc: X-Folder: \Phillip_Allen_Jan2002_1\Allen, Phillip K.\'Sent Mail X-Origin: Allen-P X-FileName: pallen (Non-Privileged).pst Here is our forecast",List(tim.belden@enron.com),phillip.allen@enron.com
"Message-ID: <15464986.1075855378456.JavaMail.evans@thyme> Date: Fri, 4 May 2001 13:51:00 -0700 (PDT) From: phillip.allen@enron.com To: john.lavorato@enron.com Subject: Re: Mime-Version: 1.0 Content-Type: text/plain; charset=us-ascii Content-Transfer-Encoding: 7bit X-From: Phillip K Allen X-To: John J Lavorato X-cc: X-bcc: X-Folder: \Phillip_Allen_Jan2002_1\Allen, Phillip K.\'Sent Mail X-Origin: Allen-P X-FileName: pallen (Non-Privileged).pst Traveling to have a business meeting takes the fun out of the trip. Especially if you have to prepare a presentation. I would suggest holding the business plan meetings here then take a trip without any formal business meetings. I would even try and get some honest opinions on whether a trip is even desired or necessary. As far as the business meetings, I think it would be more productive to try and stimulate discussions across the different groups about what is working and what is not. Too often the presenter speaks and the others are quiet just waiting for their turn. The meetings might be better if held in a round table discussion format. My suggestion for where to go is Austin. Play golf and rent a ski boat and jet ski's. Flying somewhere takes too much time.",List(john.lavorato@enron.com),phillip.allen@enron.com
"Message-ID: <24216240.1075855687451.JavaMail.evans@thyme> Date: Wed, 18 Oct 2000 03:00:00 -0700 (PDT) From: phillip.allen@enron.com To: leah.arsdall@enron.com Subject: Re: test Mime-Version: 1.0 Content-Type: text/plain; charset=us-ascii Content-Transfer-Encoding: 7bit X-From: Phillip K Allen X-To: Leah Van Arsdall X-cc: X-bcc: X-Folder: \Phillip_Allen_Dec2000\Notes Folders\'sent mail X-Origin: Allen-P X-FileName: pallen.nsf test successful. way to go!!!",List(leah.arsdall@enron.com),phillip.allen@enron.com
"Message-ID: <13505866.1075863688222.JavaMail.evans@thyme> Date: Mon, 23 Oct 2000 06:13:00 -0700 (PDT) From: phillip.allen@enron.com To: randall.gay@enron.com Subject: Mime-Version: 1.0 Content-Type: text/plain; charset=us-ascii Content-Transfer-Encoding: 7bit X-From: Phillip K Allen X-To: Randall L Gay X-cc: X-bcc: X-Folder: \Phillip_Allen_Dec2000\Notes Folders\'sent mail X-Origin: Allen-P X-FileName: pallen.nsf Randy, Can you send me a schedule of the salary and level of everyone in the scheduling group. Plus your thoughts on any changes that need to be made. (Patti S for example) Phillip",List(randall.gay@enron.com),phillip.allen@enron.com
"Message-ID: <30922949.1075863688243.JavaMail.evans@thyme> Date: Thu, 31 Aug 2000 05:07:00 -0700 (PDT) From: phillip.allen@enron.com To: greg.piper@enron.com Subject: Re: Hello Mime-Version: 1.0 Content-Type: text/plain; charset=us-ascii Content-Transfer-Encoding: 7bit X-From: Phillip K Allen X-To: Greg Piper X-cc: X-bcc: X-Folder: \Phillip_Allen_Dec2000\Notes Folders\'sent mail X-Origin: Allen-P X-FileName: pallen.nsf Let's shoot for Tuesday at 11:45.",List(greg.piper@enron.com),phillip.allen@enron.com
"Message-ID: <30965995.1075863688265.JavaMail.evans@thyme> Date: Thu, 31 Aug 2000 04:17:00 -0700 (PDT) From: phillip.allen@enron.com To: greg.piper@enron.com Subject: Re: Hello Mime-Version: 1.0 Content-Type: text/plain; charset=us-ascii Content-Transfer-Encoding: 7bit X-From: Phillip K Allen X-To: Greg Piper X-cc: X-bcc: X-Folder: \Phillip_Allen_Dec2000\Notes Folders\'sent mail X-Origin: Allen-P X-FileName: pallen.nsf Greg, How about either next Tuesday or Thursday? Phillip",List(greg.piper@enron.com),

In [0]:

display(internal_emails.count())

349813

In [0]:

validation = internal_emails.filter(
    ~has_enron_udf("to_email_list")
)
display(validation)

message,to_email_list,from_email


# Question B4

What are the top 10 domains by email count in the 'From' field, excluding emails with the Enron domain (e.g., @hotmail.com, @yahoo.com, etc.)? Include the number of emails from each domain in the answer.

In [0]:
top_domains = (
    messages_df
    .withColumn("from_email", extract_from_udf("message"))
    .filter(~col("from_email").contains("@enron.com")) # From contains @enron.com
    .groupBy("from_email")
    .count()
    .orderBy("count", ascending=False)
    .limit(10)
)

display(top_domains)

from_email,count
arsystem@mailman.enron.com,1701
exchangeinfo@nymex.com,1135
mjones7@txu.com,1063
noreply@ccomad3.uu.commissioner.com,877
soblander@carrfut.com,863
owner-nyiso_tech_exchange@lists.thebiz.net,712
sgovenar@govadv.com,619
feedback@intcx.com,607
enron_update@concureworkplace.com,523
owner-eveningmba@haas.berkeley.edu,508


%md
# Question B5

What are the top 10 pairs of sender-recipient combinations (based on 'From' and 'To' fields and treating each recipient
separately) along with the number of emails exchanged by them

# Question C1

What are the word frequencies of the top 100 words in the subject lines (using the separately provided stopwords.txt
document to remove stop words)?

# Question C2

Who are the top ten senders (based on the 'From’ field) who received no emails themselves (based on the 'To’ field)

The approach here is to find those emails which are available in the `From` field but which we can't find anywhere in the `To` field.

In [0]:
senders_df =(
    messages_df
    .withColumn("from_email", extract_from_udf("message"))
    .groupBy("from_email")
    .count()
)

recipients_df = (
    messages_df
    .withColumn("to_email_list", extract_to_udf("message"))
    .filter(col("to_email_list").isNotNull())
    .withColumn("to_email", explode("to_email_list"))
    .select("to_email")
    .distinct()
)

top_never_received = (
    senders_df
    .join(recipients_df, senders_df.from_email == recipients_df.to_email, "left_anti")
    .orderBy("count", ascending=False)
    .limit(10)
)

In [0]:
display(top_never_received)

from_email,count
no.address@enron.com,5112
noreply@ccomad3.uu.commissioner.com,877
owner-nyiso_tech_exchange@lists.thebiz.net,712
houston <.ward@enron.com>,708
legal <.taylor@enron.com>,514
owner-eveningmba@haas.berkeley.edu,508
exchange.administrator@enron.com,455
wsmith@wordsmith.org,454
fool@motleyfool.com,417
nytdirect@nytimes.com,348


# Question C3

Enron collapsed in December 2001. Calculate the number of emails exchanged each day of the year 2001 based on the ‘Date’ field and visualise this in a line graph.